# Academic Alpha Slicer — v0.5

**From alpha referee to alpha anatomy — now with model-depth survival.**

This notebook decomposes the apparent return of a systematic strategy into traded premia, implementation drag and a residual core, audits how strongly that core survives alternative inference standards, and traces an **Alpha Survival Curve** across an expanding pre-declared priced-factor span.

v0.5 keeps the v0.4c inference-policy layer unchanged and adds a second robustness dimension:

- **model/span robustness:** MKT → FF-style factors → all linear → conditional managed portfolios → traded tail factors;
- **inference robustness:** discovery → balanced → certification / full inference frontier.

These answer different questions. A strategy may be robust to standard errors but fragile to factor span, or vice versa. The survival hierarchy is fixed before looking at the alpha; non-traded nonlinear/tail-shape controls never enter it because their means are not prices. The Tail-Aware Referee remains a separate frozen certification layer.


In [ ]:
from pathlib import Path
from dataclasses import replace
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Make the local package importable when the notebook is run from the repo root.
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from alpha_slicer import AlphaSlicer, SliceConfig, make_demo_case, BLOCK_TYPES
from alpha_slicer.reporting import plot_waterfall, plot_shapley, plot_bootstrap_attribution, plot_inference_frontier, plot_alpha_survival

pd.set_option('display.width', 180)
pd.set_option('display.max_columns', 50)
import alpha_slicer
print('Academic Alpha Slicer loaded, version', alpha_slicer.__version__)

## 1. Configuration

The block order is a **presentation convention**, not a claim of causal precedence. The **pricing type** is not a choice: it is fixed by the nature of each block (traded excess returns versus non-traded functions of returns). States are lagged by `state_lag` days, so the state applied to the return of day t is known at the close of t−1.

The notebook demo uses 250 stationary-bootstrap draws for a practical `Run All`; production studies should raise this (e.g. 1,000–5,000). Stress-case illustration cells disable bootstrap when they do not use certification decisions.


In [ ]:
CFG = SliceConfig(
    annualization=252,
    hac_lags=21,
    market_col='MKT',
    nonlinear_max_order=3,
    tail_quantile=0.10,
    min_obs=500,
    state_lag=1,
    block_order=('linear', 'conditional', 'tail', 'nonlinear', 'tail_shape'),
    # v0.4c: choose this BEFORE inspecting the result. `balanced` exactly reproduces v0.4b.
    inference_profile='balanced',  # discovery | balanced | certification | custom
    inference_profile_precommitted=True,
    core_hac_lags=126,
    inference_hac_lags=(21, 63, 126),
    inference_bootstrap_reps=250,
    inference_bootstrap_mean_block=63,
    inference_bootstrap_seed=20261019,
    # v0.5: fixed model-depth path; custom spans must be nested and pre-declared.
    survival_enabled=True,
    survival_include_default_linear_spans=True,
)
display(pd.Series(BLOCK_TYPES, name='pricing type').to_frame())
CFG

## 2. Known-truth offline validation

The synthetic strategy is deliberately contaminated by several sources at once:

- static factor betas;
- extra MOM exposure after high-vol days, when the MOM premium is also higher (a conditional premium);
- a traded short-vol factor;
- a traded variance swap through which the strategy *pays* a variance risk premium;
- a small core alpha (1.0%/yr net);
- explicit transaction costs.

Every exposure is traded, so the truth is exact. This is not an empirical result; it is a functional test of whether the slicer can separate overlapping explanations.

In [ ]:
demo = make_demo_case(n=3500, seed=14)
print('Known truth / construction:')
display(demo['truth'].to_frame('value'))

slicer = AlphaSlicer(CFG)
RES = slicer.fit(
    strategy_net=demo['strategy_net'],
    strategy_gross=demo['strategy_gross'],
    factors=demo['factors'],
    benchmark=demo['benchmark'],
    tail_factors=demo['tail_factors'],
    states=demo['states'],
)
print('\nSummary:')
display(RES.summary.to_frame('value'))

## 3. Sequential alpha erosion

Read each row as the annualized intercept that remains after adding that **traded** block. The difference from the prior row is the block's **order-dependent** erosion. Non-traded blocks have no row: they are orthogonal to the traded regressors and to the constant, so they cannot move the intercept.

In [ ]:
display(RES.waterfall.round(5))
plot_waterfall(RES, 'Sequential alpha erosion — demo')
plt.show()

## 4. Order-robust block attribution

The Shapley table averages each traded block's marginal erosion across every ordering. `order_sensitivity` is the spread between the most and least favourable ordering for that block. Non-traded blocks receive exactly zero.

In [ ]:
display(RES.shapley.round(5))
print('Add-up check: Shapley blocks + core = raw net annualized mean')
print('Shapley sum:', round(float(RES.shapley.contribution_ann.sum()), 8))
print('Raw mean   :', round(float(RES.summary.raw_net_mean_ann), 8))
plot_shapley(RES, 'Shapley alpha attribution — demo')
plt.show()

## 5. Alpha Survival Curve — how deep is the alpha?

This is **model/span robustness**, not another standard-error choice. The engine starts from the raw net mean and expands only through **traded/priced returns**. With this demo the available hierarchy is `MKT → FF3 → FF3+MOM → +CONDITIONAL → +TRADED_TAIL`.

The path is not forced to be monotone. A newly added traded factor can *raise* the intercept when its premium is negative or when omitted-variable bias changes the intercept. What matters is that the final point equals the same full traded-span core used everywhere else in the Slicer.

`screen_pass` is a stage-level HAC+DSR diagnostic under the selected profile. It does **not** replace the full-model v0.4c candidate decision or the Tail-Aware Referee.

In [ ]:
cols = [
    'depth', 'family_added', 'n_priced_terms', 'alpha_ann',
    'erosion_from_prior_ann', 'survival_ratio_vs_raw',
    't', 't_threshold', 'core_dsr', 'screen_pass'
]
display(RES.alpha_survival[cols].round(5))

print('Final survival alpha :', round(float(RES.alpha_survival.iloc[-1].alpha_ann), 8))
print('Full Slicer core      :', round(float(RES.summary.core_unexplained_alpha_ann), 8))
print('Exact equality        :', np.isclose(RES.alpha_survival.iloc[-1].alpha_ann,
                                                  RES.summary.core_unexplained_alpha_ann, atol=1e-10))
print('Span fragile?         :', RES.summary.alpha_survival_span_fragile)
print('Final / raw alpha     :', round(float(RES.summary.alpha_survival_final_vs_raw), 4))
plot_alpha_survival(RES, 'v0.5 — Alpha Survival Curve')
plt.show()

## 6. Core inference and the selected policy

The point alpha is invariant to the inference profile. The selected profile changes only the uncertainty standard used for the Slicer's candidate label. `balanced` is exactly the v0.4b traded-only HAC(126) fixed-b rule. The broader `core_inference` audit is still reported separately.


In [ ]:
display(RES.core_inference.round(5))
print('Primary inference policy:', RES.summary.inference_profile,
      '| precommitted:', RES.summary.inference_profile_precommitted)
display(RES.summary[['core_unexplained_alpha_ann', 'core_alpha_t', 'core_t_critical',
                     'primary_profile_candidate_pass', 'profile_sensitive',
                     'inference_frontier_robustness_score', 'inference_fragile']].to_frame('value'))


### 6A. Power / robustness policy comparison

The primary profile must be chosen ex ante. The engine nevertheless computes all three standard policies. A discovery-only pass is evidence worth investigating, not the same claim as a balanced or certification pass. `profile_sensitive=True` records that distinction explicitly.

`inference_frontier` goes one step further: it displays the core under conventional and fixed-b HAC bandwidths plus the stationary bootstrap. The `evidence_ratio` is the statistic divided by the threshold required by that specification; 1 is the boundary.


In [ ]:
display(RES.profile_comparison.round(5))
print('Inference frontier:')
display(RES.inference_frontier.round(5))
plot_inference_frontier(RES, 'v0.4c — power / robustness frontier')
plt.show()


### 6B. A genuine policy-boundary example

The profiles are not cosmetic aliases. The synthetic series below is deliberately close to the evidence boundary: the higher-power `discovery` rule passes, while the v0.4b `balanced` standard does not. The point alpha is identical. This is exactly the kind of case that should be labelled **power-sensitive**, not silently promoted or rejected by switching estimators.


In [ ]:
from alpha_slicer import evaluate_inference_profiles
from alpha_slicer.luck import deflated_sharpe
rng = np.random.default_rng(3)
n, rho = 2500, 0.5
e = np.zeros(n)
z = rng.normal(0, 0.008, n)
for t in range(1, n):
    e[t] = rho * e[t-1] + z[t]
idx = pd.bdate_range('2010-01-01', periods=n)
y_boundary = pd.Series(e + 0.045/252, index=idx, name='boundary_alpha')
luck_boundary = deflated_sharpe(y_boundary, 1)
luck_boundary['dsr_min'] = 0.95
boundary = evaluate_inference_profiles(
    y_boundary, pd.DataFrame(index=idx), 252, 2.0, 126, None,
    luck_boundary, primary_profile='balanced')
display(boundary[['t','t_threshold','dsr','hac_pass','candidate_pass']].round(5))


### Why the zero-priced controls must stay out of the core's standard error

Take a put sold at its population expected payoff in a market with persistent volatility regimes (the suite's `peso_short_put`). The controls `m²`, `m³` and the tail terms explain most of the put's day-to-day P&L, so in the full design the residuals are small. The core's actual error, though, comes from the put's **sample mean**: how many crashes the sample happened to contain. Orthogonalization pushes that mean into the intercept, where the full-design standard error cannot see it. The cell compares both standard errors on one sample and then the coverage of the population alpha across the evaluation seeds.


In [ ]:
from alpha_slicer.montecarlo import CASES
fn, kw = CASES['peso_short_put']
CFG_FAST = replace(CFG, inference_bootstrap_reps=0)
peso = AlphaSlicer(CFG_FAST).fit(**fn(3, **kw)['inputs'])
display(peso.summary[['core_unexplained_alpha_ann', 'core_alpha_se_ann', 'core_alpha_t', 'core_t_critical',
                      'core_alpha_se_ann_v03', 'core_alpha_t_v03', 'core_se_ratio_vs_v03',
                      'diagnostic_label']].to_frame('value'))
display(peso.core_inference[['alpha_ann', 'se_ann', 't', 'critical_value', 'decision_family']].round(4))

p = Path('outputs/mc_suite_summary.csv')
if p.exists():
    ms = pd.read_csv(p, index_col=0)
    cols = ['coverage95_population_v03', 'coverage95_population', 'candidate_rate_v03', 'candidate_rate',
            'median_se_ratio_v04b_to_v03']
    print('Evaluation seeds 1000-1099 (outputs/mc_suite_summary.csv): pre-v0.4b statistic vs v0.4b on the same fits')
    display(ms.loc[ms['coverage95_population'].notna() | ms.index.isin(['regime_alpha', 'timing_skill']), cols].round(3))


## 7. Bootstrap uncertainty of the alpha slices

The stationary bootstrap resamples the **common estimation design jointly in geometric blocks** and refits every traded-block spanning regression / exact Shapley subset in each replication. Thus uncertainty in estimated betas flows into the contribution of each priced block and into the core.

Non-traded nonlinear/tail-shape controls remain zero-price by construction. The table below is therefore an uncertainty decomposition of the *priced* mean: linear premium + conditional premium + traded tail premium + core = raw net mean. Implementation drag is bootstrapped alongside it when gross P&L is supplied.

In [ ]:
display(RES.bootstrap_inference.round(5))
print('Point add-up:', round(float(RES.shapley.contribution_ann.sum()), 8),
      ' vs raw net mean:', round(float(RES.summary.raw_net_mean_ann), 8))
plot_bootstrap_attribution(RES, 'v0.4c — stationary-bootstrap uncertainty of alpha slices')
plt.show()

## 8. Does the relationship stop at linear beta?

Linear, quadratic and cubic market dependence, with HAC inference. The powers are orthogonalized to the factors, so **alpha is identical at every order by construction**. R², BIC and the highest-power t-stat show how strong the non-linear dependence is, without ever charging the raw mean of `m²` as if it were a price. Here the quadratic dependence is strong because the strategy holds a variance swap; in the full model that swap is a traded factor and spans `m²` completely (`MKT_POW2` drops out, see `dropped_columns` in the summary).

In [ ]:
display(RES.nonlinear_order.round(5))

## 9. Conditional premia and regime alphas

Two different objects:

- **Conditional premium** — exposure that rises when a lagged public state predicts a higher premium. It is captured by the traded managed portfolios `F × (z(t−1) − z̄)` in the conditional block and is **not** alpha (it is Cov(β_t, λ_t) in the four-term decomposition).
- **Regime alpha** — return that is not explained by any priced exposure but is concentrated in some states. The table below estimates it with the state on the **alpha side**: `y = a0 + a1·z(t−1) + b'X + e`, so alpha_off = a0 and alpha_on = a0 + a1, with HAC errors. A state dummy is time-varying alpha, so it is never removed as a "premium".

In the demo the core is a constant 1%/yr: alpha_on and alpha_off should not differ significantly.

In [ ]:
display(RES.conditional_alpha.round(5))

## 10. Tail diagnostics, unpriced tail-risk flags and referee hand-off

Tail-Aware-style diagnostics on the **core return**: the strategy net of every traded charge, whose mean is the core alpha. The three flags are measured after the traded blocks, so a supplied traded tail factor that prices the risk switches them off:

| flag | fires when |
|---|---|
| `short_convexity_tm` | Treynor–Mazuy quadratic loading t ≤ −2 (short volatility) |
| `excess_left_tail_beta` | extra beta in the market's left tail t ≥ 2 (short-put-like) |
| `left_tail_asymmetry` | \|ES 2.5%\| / ES 97.5% of the core return ≥ 1.5 (crash risk, market-linked or not) |

A positive, significant core that comes with any flag is labelled `core_unexplained_candidate_tail_flagged`. It goes to the referee's hidden-risk (H2) gate first, never straight to certification.

In [ ]:
display(RES.tail_diagnostics.to_frame('value').round(5))
display(RES.tail_flags.round(4))
print('Label   :', RES.summary.diagnostic_label)
print('Hand-off:', RES.summary.referee_handoff)

## 11. Non-traded exposures: what would have to be priced?

For every non-traded term the table reports:

- **coef, t** — the loading on the part not spanned by traded factors;
- **raw_mean_ann** — the regressor's raw mean (*not* a price);
- **naive_charge_ann** — what v0.1 subtracted by treating that mean as a premium;
- **breakeven_premium_ann** — the premium per unit that would drive the core to zero, also expressed as a multiple of the raw mean.

The summary also reports the core under two alternative conventions as a sensitivity: de-meaned controls, and charging raw means (v0.1).

In [ ]:
display(RES.nontraded.round(5))
display(RES.summary[['core_unexplained_alpha_ann', 'core_alt_demeaned_controls_ann',
                     'nontraded_naive_charge_ann', 'core_if_nontraded_charged_at_raw_mean_ann']].to_frame('value'))

## 12. Nested model audit

Use this table when a block appears to erase alpha: check R², HAC alpha, BIC and the condition number before interpreting the erosion economically. Alpha stays constant once only non-traded blocks are added.

In [ ]:
display(RES.nested_models.round(5))
print('\nLargest full-model coefficients by absolute t-stat:')
full_name = RES.nested_models.index[-1]
display(RES.coefficients[RES.coefficients.model == full_name]
        .assign(abs_t=lambda x: x.t.abs()).sort_values('abs_t', ascending=False)
        .drop(columns='abs_t').head(15).round(5))

## 13. Why the convention matters: four stress cases

One seed from four cases of the Monte Carlo suite, each with a known answer:

- **actuarial short put** — a put sold at its average payoff. No option premium is earned and the seller is not paid for the market risk in the put's delta, so the traded-model alpha is negative. v0.1 charged the raw means of the non-linear regressors and reported a huge positive alpha instead.
- **timing skill** — beta chosen with a noisy preview of the market. That is genuine alpha (Cov(β_t, u_t)). v0.1 turned it negative.
- **crash premium, unpriced vs traded** — a crash-insurance seller. Without a traded crash factor the premium cannot be priced: the core is large and is flagged. With the factor supplied, the core is about zero.

In [ ]:
from alpha_slicer.montecarlo import CASES

CFG_FAST = replace(CFG, inference_bootstrap_reps=0)
rows = []
for case in ['actuarial_short_put', 'timing_skill', 'crash_premium_unpriced', 'crash_premium_traded']:
    fn, kw = CASES[case]
    data = fn(3, **kw)
    r = AlphaSlicer(CFG_FAST).fit(**data['inputs'])
    s = r.summary
    rows.append({'case': case, 'core_ann': s.core_unexplained_alpha_ann, 'core_t': s.core_alpha_t,
                 'label': s.diagnostic_label, 'flags': s.tail_flags_fired,
                 'v0.1 convention core_ann': s.core_if_nontraded_charged_at_raw_mean_ann})
display(pd.DataFrame(rows).set_index('case').round(4))

## 14. Luck gate: is the core more than the best of many tries?

A core with t ≥ 2 is evidence only if it was the one thing tried. Researchers try several factor sets, state definitions and strategy variants, then report the best. Since v0.3 a core is a candidate only if it passes a **luck gate**:

- **Single specification:** the v0.4b t-stat above its fixed-b threshold (the "t ≥ 2" rule scaled by cv(b)/1.96, about 2.13) **and** a deflated Sharpe ratio ≥ 0.95 at `luck_n_trials` declared tries. With one try, the deflated Sharpe ratio is the probabilistic Sharpe ratio against zero, corrected for skewness and kurtosis.
- **Across specifications or variants:** `spec_sweep` fits every specification and gates the best core. It must pass Hansen's SPA test across all of them (stationary bootstrap, so robust to autocorrelation and to correlation between specifications), and its deflated Sharpe ratio is computed at the effective number of specifications.

The cell below reports the best of 10 zero-alpha strategy variants on the same market, which is pure selection luck.

In [ ]:
from alpha_slicer.montecarlo import case_best_of_k, LUCK_CONFIG
from alpha_slicer.luck import multi_core_gate

print('Demo core, single specification:')
display(RES.summary[['core_unexplained_alpha_ann', 'core_alpha_t', 'core_sharpe_ann', 'core_psr', 'core_dsr',
                     'luck_n_trials', 'luck_gate_pass']].to_frame('value'))

d = case_best_of_k(seed=29)   # 10 variants, none with alpha (suite seed 29)
cores, ts, t_min = {}, {}, 0.0
for name, y in d['variants'].items():
    r = AlphaSlicer(LUCK_CONFIG).fit(y, d['factors'])
    cores[name], ts[name] = r.core_returns, r.summary['core_alpha_t']
    t_min = max(t_min, r.summary['core_t_critical'])   # fixed-b threshold
g = multi_core_gate(pd.concat(cores, axis=1), pd.Series(ts), t_min=t_min, B=1000, seed=29)
print(f'Threshold {t_min:.2f}. t of the 10 zero-alpha variants:', pd.Series(ts).round(2).to_dict())
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in g.items()})

p = Path('outputs/mc_luck_summary.csv')
if p.exists():
    print('\nOver 100 seeds (outputs/mc_luck_summary.csv):')
    display(pd.read_csv(p, index_col=0)[['naive_rate', 'gated_rate', 'best_is_genuine_rate', 'median_best_t',
                                         'median_spa_p', 'median_effective_trials']])

## 15. Monte Carlo known-truth suite

The full suite runs 11 cases × 100 seeds, plus the best-of-10 luck family, against targets fixed before each evaluation run:

```bash
python -m alpha_slicer.montecarlo --seeds 100 --jobs 2 --out outputs
```

It takes about 14 minutes on two cores. The cell below reads the stored results; `outputs/MC_SUITE.md` has every target and its rationale. Truth definitions, and the changes made after smoke tests on non-evaluation seeds, are logged in the module docstring of `alpha_slicer/montecarlo.py`.

In [ ]:
from pathlib import Path
p = Path('outputs/mc_suite_summary.csv')
if p.exists():
    S = pd.read_csv(p, index_col=0)
    cols = ['median_truth', 'median_core', 'coverage95', 'candidate_rate', 'flag_rate',
            'clean_candidate_rate', 'top_label', 'median_core_v01_convention']
    display(S[cols])
    T = pd.read_csv('outputs/mc_suite_targets.csv')
    print(f"Targets passed: {int(T['pass'].sum())}/{len(T)}")
    display(T[~T['pass']])
else:
    print('Run the suite first (see the command above).')

### Persistence stress test (v0.4, rerun in v0.4b)

A separate zero-alpha stress test isolates the standard-error problem: the population alpha is zero, but the conditional mean follows a persistent two-state Markov chain. v0.4 reported 7.5% false rejections for HAC(126) and a 126-day stationary bootstrap from 40 seeds per world. With 400 seeds per world the same design gives about 16–17%: a long bandwidth halves the error of HAC(21) but stays far above the nominal 5%, and 200-day regimes push it up again. Score prewhitening does **not** help.

The table is `outputs/inference_validation_summary.csv` (false-rejection rate with Wilson 95% intervals).


In [ ]:
p_inf = Path('outputs/inference_validation_summary.csv')
if p_inf.exists():
    iv = pd.read_csv(p_inf)
    display(iv.pivot(index='world', columns='method', values='false_reject_rate').round(3))
else:
    print('Run: python -m alpha_slicer.inference_validation --seeds 400 --start-seed 1000 --out outputs')


## 16. Input contract for any academic alpha

The slicer itself does **not** care how the strategy was generated. The minimum contract is:

```python
strategy_net : pd.Series       # daily net excess returns
factors      : pd.DataFrame    # daily TRADED factor excess returns; include MKT or pass benchmark
benchmark    : pd.Series       # market/benchmark return
```

Optional but high-value inputs:

```python
strategy_gross : pd.Series     # separates implementation drag
tail_factors   : pd.DataFrame  # TRADED tail/convexity P&L (variance swap, put-writing index, crash factor):
                               # the only way to price a non-linear premium; see alpha_slicer.variance_swap_pnl
states         : pd.DataFrame  # ex-ante states AS OBSERVED AT THE CLOSE; the engine lags them by state_lag
```

Declare your search: `SliceConfig(luck_n_trials=N)` with N = specifications × strategy variants tried before reporting this one, or run `spec_sweep` over the specifications you considered.

**Important:** `core_unexplained_alpha` means unexplained by the **traded** factors supplied. It is not automatically causal or genuine alpha. A flagged core goes to the referee's hidden-risk gate, and a clean core still needs the referee's luck gate.

## 17. Optional live case A — current ETF Factor Momentum repo

This adapter reuses the current repository's `core.py`, downloads the five factor ETFs plus SPY, rebuilds `MomTop1_80_20`, and slices its **net** P&L. The static ETF sleeves + SPY form the linear block, which directly asks whether the timing rule adds anything beyond static factor exposure.

Leave `RUN_LIVE_MOM = False` for the offline reproducible notebook.

In [ ]:
RUN_LIVE_MOM = False
MOM_REPO = Path('../cross-sectional-factor-momentum-main')

if RUN_LIVE_MOM:
    from alpha_slicer.adapters import run_factor_momentum_live
    mom = run_factor_momentum_live(MOM_REPO, start='2000-01-01', variant='6-1')
    MOM_RES = slicer.fit(mom['strategy_net'], mom['factors'], mom['benchmark'],
                         strategy_gross=mom['strategy_gross'])
    display(MOM_RES.summary.to_frame('value'))
    display(MOM_RES.shapley.round(5))
    display(MOM_RES.nonlinear_order.round(5))
    display(MOM_RES.tail_flags.round(4))
else:
    print('Live Factor Momentum run skipped (set RUN_LIVE_MOM=True).')

## 18. Optional live case B — current Tail-Aware S&P100 residual momentum

Since v0.3 this adapter loads the referee's own F0 engine (the `02_f0_sp100_public_data` notebook, or its core `.py` export) and calls the referee's functions — data loader, residualization ladder, signals, backtest — at the referee's verdict level. The slicer therefore explains exactly the P&L the referee judges. On the referee's offline panel the two are identical to machine precision.

The Tail-Aware **referee** and the Slicer answer different questions:

- referee: *should this signal be certified?*
- slicer: *what explains the apparent return that exists?*

Leave the flag off for the offline run.

In [ ]:
RUN_LIVE_TAIL = False
TAIL_REPO = Path('../tail-aware-alpha-referee-main')

if RUN_LIVE_TAIL:
    from alpha_slicer.adapters import run_tailaware_sp100_live
    ta = run_tailaware_sp100_live(TAIL_REPO, start='2007-01-01', end='2025-01-01')
    TA_RES = slicer.fit(ta['strategy_net'], ta['factors'], ta['benchmark'],
                        strategy_gross=ta['strategy_gross'])
    display(TA_RES.summary.to_frame('value'))
    display(TA_RES.shapley.round(5))
    display(TA_RES.conditional_alpha.round(5))
    display(TA_RES.tail_diagnostics.to_frame('value').round(5))
    display(TA_RES.tail_flags.round(4))
else:
    print('Live Tail-Aware-style S&P100 run skipped (set RUN_LIVE_TAIL=True).')

## Research interpretation template

For each strategy, report in this order:

1. **Raw/gross → net:** implementation drag.
2. **Sequential waterfall + Shapley:** economic attribution and order sensitivity.
3. **Alpha Survival Curve:** how much intercept survives MKT / FF-style / managed / traded-tail priced spans; report span fragility separately from inference fragility.
4. **Nonlinear order / regime anatomy / tail diagnostics:** shape and state dependence that may not itself carry a price.
5. **Core point estimate:** intercept after the richest supplied traded span.
6. **Primary inference profile:** `discovery`, `balanced`, `certification` or `custom`, and whether it was precommitted.
7. **Profile comparison + inference frontier:** does the same final core survive stricter uncertainty standards, or is it power-sensitive?
8. **Luck gate:** DSR/SPA at the number of trials actually spent.
9. **Referee hand-off:** Tail-Aware remains the separate hidden-risk / alpha-certification layer.

The engine now answers three distinct questions: **what is the alpha made of?**, **how deep does it survive as the pricing model expands?**, and **how strong must the inferential standard become before the residual core disappears?**
